# Benchmark: Constraint Mode Comparison

Compares the four constraint configurations on the same deformation fields:

| Mode | Flags |
|------|-------|
| **Jacobian only** | default |
| **Jacobian + Shoelace** | `enforce_shoelace=True` |
| **Jacobian + Injectivity** | `enforce_injectivity=True` |
| **All constraints** | both `True` |

Metrics: runtime, L2 error, final min Jdet, SLSQP iterations (outer), and
whether all negative Jacobians were eliminated.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    iterative_serial,
    iterative_parallel,
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
    shoelace_det2D,
)
from dvfopt.jacobian import _monotonicity_diffs_2d, _diagonal_monotonicity_diffs_2d
from dvfopt.jacobian.intersection import has_quad_self_intersections
from test_cases import SYNTHETIC_CASES, make_deformation, make_random_dvf
from dvfopt.viz import plot_deformations, plot_grid_before_after
from benchmark_utils import (
    get_output_dir, save_figure, save_results_csv, save_summary_json, log_run_header, log_run_footer, results_to_rows, show_and_save, reset_figure_counter,
)


In [ ]:
METHOD = "slsqp"
NOTEBOOK_NAME = "constraint-modes"
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base="../../output")
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)


## Constraint modes

In [ ]:
# ── Solver ───────────────────────────────────────────────────────────────────
# Switch between iterative_serial (safer, single-process) and
# iterative_parallel (faster on multi-core, avoids Windows spawn overhead
# for single-window cases).
SOLVER = iterative_serial
# SOLVER = iterative_parallel

# ── Jdet colormap cap ─────────────────────────────────────────────────────
# Values beyond ±JDET_VMAX are clamped so extreme outliers don't wash out
# the colour scale around zero.
JDET_VMAX = 10

# ── Injectivity threshold ─────────────────────────────────────────────────
# Explicit threshold bypasses the adaptive doubling loop (which reruns the
# full solver up to max_doublings+1 times).  Set to None to use adaptive.
INJECTIVITY_THRESHOLD = 0.3


In [ ]:
MODES = {
    "Jacobian only":   {"enforce_shoelace": False, "enforce_injectivity": False},
    "Jac + Shoelace":  {"enforce_shoelace": True,  "enforce_injectivity": False},
    "Jac + Injectivity": {"enforce_shoelace": False, "enforce_injectivity": True,
                          "injectivity_threshold": INJECTIVITY_THRESHOLD},
    "All constraints": {"enforce_shoelace": True,  "enforce_injectivity": True,
                        "injectivity_threshold": INJECTIVITY_THRESHOLD},
}


## Helper: run one test case across all modes

In [ ]:
def _injectivity_stats(phi):
    """Compute shoelace, h/v/d1/d2 monotonicity violation counts and global intersection."""
    shoe = np.squeeze(shoelace_det2D(phi))
    h_m, v_m = _monotonicity_diffs_2d(phi[0], phi[1])
    d1, d2   = _diagonal_monotonicity_diffs_2d(phi[0], phi[1])
    # Global check: any non-adjacent quad cells intersect geometrically?
    # O(n^2) — slow on large grids but definitive.
    global_intersect = has_quad_self_intersections(phi)
    return dict(
        n_neg_shoe       = int((shoe[1:-1, 1:-1] <= 0).sum()),
        n_h_viol         = int((h_m[1:-1, 1:-1] <= 0).sum()),
        n_v_viol         = int((v_m[1:-1, 1:-1] <= 0).sum()),
        n_d1_viol        = int((d1[1:-1, 1:-1] <= 0).sum()),
        n_d2_viol        = int((d2[1:-1, 1:-1] <= 0).sum()),
        global_intersect = global_intersect,
    )


def run_case(deformation_i, label, n_runs=1):
    """Run all constraint modes on a single deformation field."""
    phi_init = np.stack([deformation_i[-2, 0], deformation_i[-1, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg_init = int((jac_init <= 0).sum())
    H, W = deformation_i.shape[-2:]

    print(f"\n{'='*80}")
    print(f"  {label}  |  {H}x{W}  |  Initial neg-Jdet: {n_neg_init}")
    print(f"{'='*80}")

    results = {}
    for mode_name, flags in MODES.items():
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            phi = SOLVER(
                deformation_i.copy(), verbose=0, **flags
            )
            times.append(time.perf_counter() - t0)

        jac_final = jacobian_det2D(phi)
        final_neg = int((jac_final <= 0).sum())
        final_min = float(jac_final.min())
        l2_err = float(np.sqrt(np.sum((phi - phi_init) ** 2)))
        avg_t = np.mean(times)
        inj = _injectivity_stats(phi)

        results[mode_name] = {
            "time": avg_t,
            "final_neg": final_neg,
            "final_min": final_min,
            "l2_err": l2_err,
            "phi": phi,
            **inj,
        }

        jac_ok  = "OK" if final_neg == 0                  else f"FAIL({final_neg})"
        shoe_ok = "OK" if inj["n_neg_shoe"] == 0          else f"FAIL({inj['n_neg_shoe']})"
        h_ok    = "OK" if inj["n_h_viol"] == 0            else f"FAIL({inj['n_h_viol']})"
        v_ok    = "OK" if inj["n_v_viol"] == 0            else f"FAIL({inj['n_v_viol']})"
        d1_ok   = "OK" if inj["n_d1_viol"] == 0           else f"FAIL({inj['n_d1_viol']})"
        d2_ok   = "OK" if inj["n_d2_viol"] == 0           else f"FAIL({inj['n_d2_viol']})"
        g_ok    = "OK" if not inj["global_intersect"]     else "INTERSECT"
        print(f"  {mode_name:<20s}  {avg_t:8.2f}s  "
              f"neg={final_neg:3d}  min_jdet={final_min:+.6f}  L2={l2_err:.4f}  "
              f"[Jac:{jac_ok}  Shoe:{shoe_ok}  h:{h_ok}  v:{v_ok}  "
              f"d1:{d1_ok}  d2:{d2_ok}  glob:{g_ok}]")

        # Grid visualisation immediately after each mode
        plot_grid_before_after(deformation_i, phi, title=f"{label} — {mode_name}",
                               jdet_vmax=JDET_VMAX)

    return results


---
## Test Cases

In [ ]:
all_results = {}

# --- Synthetic correspondence cases ---
for key in ["01a_10x10_crossing", "01c_20x40_edges", "03d_20x20_crossing"]:
    deformation, ms, fs = make_deformation(key)
    title = SYNTHETIC_CASES[key]["title"]
    all_results[title] = run_case(deformation, title)

In [ ]:
# --- Random DVF cases ---
for key in ["01e_20x20_random_spirals", "01f_20x20_random_seed_42"]:
    deformation = make_random_dvf(key)
    from test_cases import RANDOM_DVF_CASES
    title = RANDOM_DVF_CASES[key]["title"]
    all_results[title] = run_case(deformation, title)

In [ ]:
# --- Real-data slice (if available) ---
import os
real_path = "../../../data/test_cases/02b_64x91_slice200.npy"
if os.path.exists(real_path):
    deformation = np.load(real_path)
    all_results["Real 64x91"] = run_case(deformation, "Real 64x91 slice 200")
else:
    print(f"Skipping real data â€” {real_path} not found")

---
## Summary Table

In [ ]:
mode_names = list(MODES.keys())

print(f"{'Test Case':<28s}", end="")
for m in mode_names:
    print(f"  {m:>20s}", end="")
print()
print("-" * (28 + 22 * len(mode_names)))

for label, results in all_results.items():
    # Time row
    print(f"{label:<28s}", end="")
    for m in mode_names:
        r = results[m]
        print(f"  {r['time']:>17.2f}s  ", end="")
    print("  (time)")

    # L2 row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        print(f"  {r['l2_err']:>18.4f} ", end="")
    print("  (L2)")

    # Min Jdet row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        print(f"  {r['final_min']:>+18.6f}", end="")
    print("  (min Jdet)")

    # Shoelace row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        v = r['n_neg_shoe']
        s = 'OK' if v == 0 else str(v)
        print(f"  {s:>18s} ", end="")
    print("  (Shoelace viol)")

    # h-mono row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        v = r['n_h_viol']
        s = 'OK' if v == 0 else str(v)
        print(f"  {s:>18s} ", end="")
    print("  (h-mono viol)")

    # v-mono row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        v = r['n_v_viol']
        s = 'OK' if v == 0 else str(v)
        print(f"  {s:>18s} ", end="")
    print("  (v-mono viol)")

    # d1-mono row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        v = r['n_d1_viol']
        s = 'OK' if v == 0 else str(v)
        print(f"  {s:>18s} ", end="")
    print("  (d1-mono viol)")

    # d2-mono row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        v = r['n_d2_viol']
        s = 'OK' if v == 0 else str(v)
        print(f"  {s:>18s} ", end="")
    print("  (d2-mono viol)")

    # Global intersection row
    print(f"{'':28s}", end="")
    for m in mode_names:
        r = results[m]
        s = 'INTERSECT' if r['global_intersect'] else 'OK'
        print(f"  {s:>18s} ", end="")
    print("  (Glob intersect)")

    print()


## Plots

In [ ]:
# --- Bar chart: runtime by mode ---
case_labels = list(all_results.keys())
x = np.arange(len(case_labels))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(mode_names):
    times = [all_results[c][m]["time"] for c in case_labels]
    ax.bar(x + i * width, times, width, label=m)

ax.set_ylabel("Time (s)")
ax.set_title("Runtime by Constraint Mode")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(case_labels, rotation=25, ha="right")
ax.legend()
plt.tight_layout()
show_and_save(OUTPUT_DIR)


In [ ]:
# --- Bar chart: L2 error by mode ---
fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(mode_names):
    l2s = [all_results[c][m]["l2_err"] for c in case_labels]
    ax.bar(x + i * width, l2s, width, label=m)

ax.set_ylabel("L2 Error")
ax.set_title("L2 Deviation by Constraint Mode")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(case_labels, rotation=25, ha="right")
ax.legend()
plt.tight_layout()
show_and_save(OUTPUT_DIR)


In [ ]:
# --- Scatter: runtime vs L2 (Pareto front) ---
fig, ax = plt.subplots(figsize=(8, 6))
markers = ["o", "s", "^", "D"]

for i, m in enumerate(mode_names):
    times = [all_results[c][m]["time"] for c in case_labels]
    l2s = [all_results[c][m]["l2_err"] for c in case_labels]
    ax.scatter(times, l2s, marker=markers[i], s=80, label=m, zorder=3)

ax.set_xlabel("Time (s)")
ax.set_ylabel("L2 Error")
ax.set_title("Runtime vs L2 Error â€” Constraint Modes")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
show_and_save(OUTPUT_DIR)


In [ ]:
# --- Heatmap: min Jdet by test case x mode ---
min_jdet_matrix = np.array([
    [all_results[c][m]["final_min"] for m in mode_names]
    for c in case_labels
])

fig, ax = plt.subplots(figsize=(10, max(3, 0.6 * len(case_labels))))
im = ax.imshow(min_jdet_matrix, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(mode_names)))
ax.set_xticklabels(mode_names, rotation=25, ha="right")
ax.set_yticks(range(len(case_labels)))
ax.set_yticklabels(case_labels, fontsize=9)
for i in range(len(case_labels)):
    for j in range(len(mode_names)):
        val = min_jdet_matrix[i, j]
        color = "white" if val < 0 else "black"
        ax.text(j, i, f"{val:+.4f}", ha="center", va="center", fontsize=8, color=color)
fig.colorbar(im, ax=ax, label="Min Jacobian determinant")
ax.set_title("Final Min Jdet -- Test Case x Constraint Mode")
plt.tight_layout()
show_and_save(OUTPUT_DIR)


In [ ]:
# --- Remaining neg-Jdet pixels by mode (convergence success) ---
neg_matrix = np.array([
    [all_results[c][m]["final_neg"] for m in mode_names]
    for c in case_labels
])

fig, ax = plt.subplots(figsize=(10, max(3, 0.6 * len(case_labels))))
cmap = plt.cm.colors.ListedColormap(["#2ecc71", "#e74c3c"])
bounds = [-0.5, 0.5, neg_matrix.max() + 0.5]
norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

im = ax.imshow(neg_matrix, cmap=cmap, norm=norm, aspect="auto")
ax.set_xticks(range(len(mode_names)))
ax.set_xticklabels(mode_names, rotation=25, ha="right")
ax.set_yticks(range(len(case_labels)))
ax.set_yticklabels(case_labels, fontsize=9)
for i in range(len(case_labels)):
    for j in range(len(mode_names)):
        val = neg_matrix[i, j]
        ax.text(j, i, str(val), ha="center", va="center", fontsize=10,
                fontweight="bold", color="white")
ax.set_title("Remaining Neg-Jdet Pixels (green=0, red=failed)")
plt.tight_layout()
show_and_save(OUTPUT_DIR)


In [ ]:
# --- Runtime overhead relative to Jacobian-only baseline ---
baseline_mode = "Jacobian only"
other_modes = [m for m in mode_names if m != baseline_mode]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(case_labels))
width = 0.25

for i, m in enumerate(other_modes):
    overhead = []
    for c in case_labels:
        base_t = all_results[c][baseline_mode]["time"]
        mode_t = all_results[c][m]["time"]
        overhead.append(mode_t / base_t if base_t > 0 else 1.0)
    ax.bar(x + i * width, overhead, width, label=m)

ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="Baseline (Jac only)")
ax.set_xticks(x + width)
ax.set_xticklabels(case_labels, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Runtime multiplier vs Jacobian-only")
ax.set_title("Constraint Overhead (1.0 = same as Jacobian-only)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
show_and_save(OUTPUT_DIR)


In [ ]:
# --- Save results ---
if 'results' in dir() and isinstance(results, dict) and results:
    rows, cols = results_to_rows(results)
    save_results_csv(rows, cols, OUTPUT_DIR)
    summary = log_run_footer(summary, results)
    save_summary_json(summary, OUTPUT_DIR)
elif 'mag_results' in dir():
    rows, cols = results_to_rows(mag_results)
    save_results_csv(rows, cols, OUTPUT_DIR, name='results_magnitude')
    if 'density_results' in dir():
        rows2, cols2 = results_to_rows(density_results)
        save_results_csv(rows2, cols2, OUTPUT_DIR, name='results_density')
    combined = {**mag_results, **density_results} if 'density_results' in dir() else mag_results
    summary = log_run_footer(summary, combined)
    save_summary_json(summary, OUTPUT_DIR)
else:
    save_summary_json(summary, OUTPUT_DIR)
    print('No results dict found; saved summary only.')
